#Indications :

On est déjà au 3ème et dernier checkpoint.

Les parties principales sont indépendants. Tu peux donc les effectuer dans l'ordre que tu préfères.

Essaye de finir ce checkpoint dans la journée. Si tu n'as pas tout fini, ce n'est pas grave. Tu pourras le reprendre plus tard, tu indiqueras simplement en commentaire ce que tu as fait a posteriori.


Évidemment chacun a ses forces et ses faiblesses, donc fait ce que tu peux. **Cependant, si tu vois que tu restes bloqué, passe à l'exercice suivant.** Tu pourras revenir à la fin sur ce que tu n'as pas fini.

Il y a deux questions bonus à la fin. Ce sont des exercices pour aller plus loin. Ils sont assez stimulants pour de l'analyse.

**P.S : Lis très bien les consignes, les exercices sont guidés, si tu sens que tu es allé très loin dans l'exercice, relis ta consigne, tu as surement raté un détail.**


# Regex

Un client veut que tu créés des regex pour trouver ces infos dans un texte.

- Qui valide la plupart des formats d'emails standards
- Qui valide une date au format AAAA-MM-JJ
- Qui valide un code postal français standard à 5 chiffres


PS : Aucune des regex ne doit provenir du site [Synapse Learners Regex](https://patrick-wampe.github.io/Synapse-Learners/regex.html)



In [7]:
import re

# 1) Valide la plupart des formats d'emails standards
regex_email = r"\S+@\S+\.\S+"

# 2) Valide une date au format AAAA-MM-JJ (mois 01-12, jour 01-31)
regex_date = r"\d{4}-\d{2}-\d{2}"

# 3) Valide un code postal français standard à 5 chiffres (départements 01 à 98)
regex_cp = r"^0[1-9]|[1-8]\d|9[0-8]$"

# ==== Tests ====
emails = ["test@exemple.com", "j.dupont+wcs@mail-server.fr", "invalide@", "@nope.com", "a@b.fr", "fab-paris@job.co.fr"]
dates = ["2026-06-15", "2026-13-01", "2026-6-1", "1999-10-31", "2026-02-30 "]
codes = ["75001", "13008", "97400", "00999", "ABCDE", "750011", "BP1200"]

print("Emails :")
for e in emails:
    print(f"  {e:32} -> {bool(re.match(regex_email, e))}")

print("\nDates AAAA-MM-JJ :")
for d in dates:
    print(f"  {d:16} -> {bool(re.match(regex_date, d))}")

print("\nCodes postaux FR :")
for c in codes:
    print(f"  {c:10} -> {bool(re.match(regex_cp, c))}")

Emails :
  test@exemple.com                 -> True
  j.dupont+wcs@mail-server.fr      -> True
  invalide@                        -> False
  @nope.com                        -> False
  a@b.fr                           -> True
  fab-paris@job.co.fr              -> True

Dates AAAA-MM-JJ :
  2026-06-15       -> True
  2026-13-01       -> True
  2026-6-1         -> False
  1999-10-31       -> True
  2026-02-30       -> True

Codes postaux FR :
  75001      -> True
  13008      -> True
  97400      -> False
  00999      -> False
  ABCDE      -> False
  750011     -> True
  BP1200     -> False


# Scraping

Un éditeur de livres souhaite que vous scrapiez le site web [**Book to Scrape**](https://books.toscrape.com/)

**Mission 1**

L'objectif est de récupérer les informations suivantes pour chaque livre :     
- Le titre du livre
- Son prix que vous convertirez en euros

L'idée est donc de récupérer les informations des livres de toutes les pages, pas juste la 1ère. Fais en un dataframe avec 2 colonnes.


In [22]:
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

lien_du_site = "https://books.toscrape.com/"

# Taux de conversion approximatif livre sterling -> euro
taux_livre_eur = 1.16


def prix_en_euros(texte_prix):
    # Convertit un prix type '£51.77' en euros (float arrondi à 2 décimales)
    montant = float(re.search(r"[\d.]+", texte_prix).group())
    return round(montant * taux_livre_eur, 2)


In [24]:
# Mission 1 : titre + prix en euros de TOUS les livres (50 pages du catalogue)
titres = []
prix = []

for page in range(1, 51):
    url = lien_du_site + f"catalogue/page-{page}.html"
    soup = BeautifulSoup(requests.get(url).content, "html.parser", )

    for livre in soup.select("article.product_pod"):
        titres.append(livre.h3.a["title"])
        prix.append(prix_en_euros(livre.select_one("p.price_color").text))

livres_m1 = pd.DataFrame({"titre": titres, "prix_euros": prix})
#print("Nombre de livres :", livres_m1.shape[0])
livres_m1.head(10)


,titre,prix_euros
0,A Light in the Attic,60.05
1,Tipping the Velvet,62.34
2,Soumission,58.12
3,Sharp Objects,55.47
4,Sapiens: A Brief History of Humankind,62.91
5,The Requiem Red,26.27
6,The Dirty Little Secrets of Getting Your Dream...,38.67
7,The Coming Woman: A Novel Based on the Life of...,20.80
8,The Boys in the Boat: Nine Americans and Their...,26.22
9,The Black Maria,60.49


**Mission 2**

Idéalement le client souhaiterai ranger les livres par catégorie. Il faudrait donc trouver un moyen d'afficher aussi la catégorie du livre comme une colonne du dataframe. Donc on aura la catégorie, le titre du livre, son prix en euros et si il est en stock ou pas

In [4]:
# Mission 2 : catégorie + titre + prix (€) + disponibilité
# On parcourt chaque catégorie du menu latéral (la catégorie est ainsi connue)
soup_accueil = BeautifulSoup(requests.get(lien_du_site).content, "html.parser")
categories = {
    a.text.strip(): urljoin(lien_du_site, a["href"])
    for a in soup_accueil.select("div.side_categories ul li ul li a")
}
print(len(categories), "catégories trouvées")

lignes = []
for nom_categorie, url_categorie in categories.items():
    url = url_categorie
    while url:  # on suit la pagination de la catégorie tant qu'il y a une page suivante
        soup = BeautifulSoup(requests.get(url).content, "html.parser")
        for livre in soup.select("article.product_pod"):
            dispo = livre.select_one("p.instock.availability").text.strip()
            lignes.append({
                "categorie": nom_categorie,
                "titre": livre.h3.a["title"],
                "prix_euros": prix_en_euros(livre.select_one("p.price_color").text),
                "en_stock": "in stock" in dispo.lower(),
            })
        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

livres_m2 = pd.DataFrame(lignes)
print("Nombre de livres :", livres_m2.shape[0])
livres_m2.head()


50 catégories trouvées


Nombre de livres : 1000


,categorie,titre,prix_euros,en_stock
0,Travel,It's Only the Himalayas,52.85,True
1,Travel,Full Moon over Noah’s Ark: An Odyssey to Mount...,57.83,True
2,Travel,See America: A Celebration of Our National Par...,57.18,True
3,Travel,Vagabonding: An Uncommon Guide to the Art of L...,43.22,True
4,Travel,Under the Tuscan Sun,43.68,True


**Mission 3**

Le client voudrait maintenant aussi la description du film et le nombre en stock en plus des éléments de la mission 2

In [5]:
# Mission 3 : on ajoute la description et le nombre exact en stock.
# Ces infos ne sont que sur la page de détail de chaque livre -> on visite chaque fiche.
from concurrent.futures import ThreadPoolExecutor

# 1) Récupérer l'URL de la fiche de chaque livre (en gardant sa catégorie)
livres_a_visiter = []
for nom_categorie, url_categorie in categories.items():
    url = url_categorie
    while url:
        soup = BeautifulSoup(requests.get(url).content, "html.parser")
        for livre in soup.select("article.product_pod"):
            url_detail = urljoin(url, livre.h3.a["href"])
            livres_a_visiter.append((nom_categorie, url_detail))
        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

print(len(livres_a_visiter), "fiches à visiter...")


def scraper_fiche(infos):
    nom_categorie, url_detail = infos
    soup = BeautifulSoup(requests.get(url_detail).content, "html.parser")

    dispo = soup.select_one("p.instock.availability").text.strip()
    recherche = re.search(r"\((\d+) available\)", dispo)
    nb_stock = int(recherche.group(1)) if recherche else 0

    bloc_desc = soup.select_one("#product_description ~ p")  # paragraphe juste après le titre "Product Description"

    return {
        "categorie": nom_categorie,
        "titre": soup.h1.text.strip(),
        "prix_euros": prix_en_euros(soup.select_one("p.price_color").text),
        "en_stock": "in stock" in dispo.lower(),
        "nb_stock": nb_stock,
        "description": bloc_desc.text.strip() if bloc_desc else "",
    }


# 2) Visiter les fiches en parallèle (1000 pages -> beaucoup plus rapide)
with ThreadPoolExecutor(max_workers=16) as executor:
    resultats = list(executor.map(scraper_fiche, livres_a_visiter))

livres_m3 = pd.DataFrame(resultats)
print("Dimensions :", livres_m3.shape)
livres_m3.head()


1000 fiches à visiter...


Dimensions : (1000, 6)


,categorie,titre,prix_euros,en_stock,nb_stock,description
0,Travel,It's Only the Himalayas,52.85,True,19,"“Wherever you go, whatever you do, just . . . ..."
1,Travel,Full Moon over Noah’s Ark: An Odyssey to Mount...,57.83,True,15,Acclaimed travel writer Rick Antonson sets his...
2,Travel,See America: A Celebration of Our National Par...,57.18,True,14,To coincide with the 2016 centennial anniversa...
3,Travel,Vagabonding: An Uncommon Guide to the Art of L...,43.22,True,8,With a new foreword by Tim Ferriss •There’s no...
4,Travel,Under the Tuscan Sun,43.68,True,7,A CLASSIC FROM THE BESTSELLING AUTHOR OF UNDER...


# Geocoding

L'ensemble de données suivant répertorie une sélection des meilleurs restaurants de Paris, à des prix très abordables (moins de 15 euros par menu en moyenne).

In [6]:
import pandas as pd
import requests

In [7]:
food_paris = pd.read_csv("https://raw.githubusercontent.com/WildCodeSchool/wilddata/main/food.csv").drop(columns = "Unnamed: 0")

In [8]:
food_paris.head()

,nom,adresse,code postal
0,Kodawari Tsukiji,12 Rue de Richelieu,75001 Paris
1,Café Lai’Tcha,7 Rue du Jour,75001 Paris
2,Pizz'Aria,55 Rue Montmartre,75002 Paris
3,M La Vie,85 Rue Montmartre,75002 Paris
4,Road Trip,36 Rue Poissonnière,75002 Paris


In [9]:
food_paris["code postal"].value_counts()

code postal
75002 Paris    5
75009 Paris    4
75010 Paris    4
75011 Paris    3
75001 Paris    2
75003 Paris    2
75004 Paris    2
75006 Paris    2
75008 Paris    2
75007 Paris    1
75014 Paris    1
75017 Paris    1
Name: count, dtype: int64

In [10]:
food_paris.info()

<class 'pandas.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   nom          29 non-null     str  
 1   adresse      29 non-null     str  
 2   code postal  29 non-null     str  
dtypes: str(3)
memory usage: 2.0 KB


In [11]:
url = "https://data.geopf.fr/geocodage/search?q=73 Avenue de Paris Saint-Mandé"

requests.get(url)

<Response [200]>

In [12]:
requests.get(url).json()

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.424573, 48.845726]},
   'properties': {'label': '73 Avenue de Paris 94160 Saint-Mandé',
    'score': 0.9651045454545454,
    'housenumber': '73',
    'id': '94067_7115_00073',
    'banId': '7b81888d-7f55-4f01-b048-9cdb47a674fb',
    'name': '73 Avenue de Paris',
    'postcode': '94160',
    'citycode': '94067',
    'x': 657770.33,
    'y': 6860785.03,
    'city': 'Saint-Mandé',
    'context': '94, Val-de-Marne, Île-de-France',
    'type': 'housenumber',
    'importance': 0.61615,
    'depcode': '94',
    'street': 'Avenue de Paris',
    '_type': 'address'}},
  {'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.406466, 48.844807]},
   'properties': {'label': '73 Avenue de Saint-Mandé 75012 Paris',
    'score': 0.8353045454545455,
    'housenumber': '73',
    'id': '75112_8682_00073',
    'name': '73 Avenue de Saint-Mandé',
    'postcode': '75012',
    'c

In [13]:
data = requests.get(url).json()

In [14]:
def recuperationLongEtLat(url):
  try:
    data = requests.get(url).json()
    longitude = data["features"][0]["geometry"]["coordinates"][1]
    latitude = data["features"][0]["geometry"]["coordinates"][0]
    return longitude, latitude
  except:
    print("L'url n'a pas marché")

In [15]:
"https://data.geopf.fr/geocodage/search?q={mon adresse}&postcode={mon code postal}"

'https://data.geopf.fr/geocodage/search?q={mon adresse}&postcode={mon code postal}'

In [16]:
coordonnees = []
for ligne in range(0, food_paris.shape[0]):
  # Récupération de l'adresse = food_paris.loc[ligne, 'adresse']
  # Récupération du code postal = food_paris.loc[ligne, 'code postal'].split()[0]

  url = f"https://data.geopf.fr/geocodage/search?q={food_paris.loc[ligne, 'adresse']}&postcode={food_paris.loc[ligne, 'code postal'].split()[0]}"
  coordonnees.append(recuperationLongEtLat(url))

food_paris["coordonnees"] = coordonnees

In [17]:
food_paris

,nom,adresse,code postal,coordonnees
0,Kodawari Tsukiji,12 Rue de Richelieu,75001 Paris,"(48.864374, 2.336258)"
1,Café Lai’Tcha,7 Rue du Jour,75001 Paris,"(48.86355, 2.344247)"
2,Pizz'Aria,55 Rue Montmartre,75002 Paris,"(48.865983, 2.344484)"
3,M La Vie,85 Rue Montmartre,75002 Paris,"(48.867596, 2.343757)"
4,Road Trip,36 Rue Poissonnière,75002 Paris,"(48.869951, 2.34794)"
5,Rolls,29 Rue des Jeuneurs,75002 Paris,"(48.869541, 2.344421)"
6,Qasti Shawarma,214 Rue Saint-Martin,75003 Paris,"(48.863752, 2.35287)"
7,The Brooklyn Pizzeria,33 Bd Beaumarchais,75003 Paris,"(48.856, 2.368186)"
8,La Baguette du relais,10 Rue des Archives,75004 Paris,"(48.857587, 2.354532)"
9,Olive & Thym,60 Rue Quincampoix,75004 Paris,"(48.861576, 2.35082)"


Affiche maintenant ces restaurants sur une carte, en utilisant la bibliothèque `folium`.

In [18]:
food_paris.head()

,nom,adresse,code postal,coordonnees
0,Kodawari Tsukiji,12 Rue de Richelieu,75001 Paris,"(48.864374, 2.336258)"
1,Café Lai’Tcha,7 Rue du Jour,75001 Paris,"(48.86355, 2.344247)"
2,Pizz'Aria,55 Rue Montmartre,75002 Paris,"(48.865983, 2.344484)"
3,M La Vie,85 Rue Montmartre,75002 Paris,"(48.867596, 2.343757)"
4,Road Trip,36 Rue Poissonnière,75002 Paris,"(48.869951, 2.34794)"


In [19]:
import folium

# Carte centrée sur Paris
carte = folium.Map(location=[48.8566, 2.3522], zoom_start=13)

for _, restaurant in food_paris.iterrows():
    coords = restaurant["coordonnees"]
    if coords:  # certaines adresses peuvent ne pas être géocodées
        folium.Marker(
            location=[coords[0], coords[1]],  # (latitude, longitude)
            popup=f"{restaurant['nom']} — {restaurant['adresse']}",
            tooltip=restaurant["nom"],
            icon=folium.Icon(color="red", icon="cutlery", prefix="fa"),
        ).add_to(carte)

carte


# Optionnel

## NLP - Classification d'analyse des sentiments - 2h

Définissez `X` qui ne contiendra que la colonne `text`et `y` qui sera la colonne `sentiment`.

In [20]:
# Section NLP non traitée : le notebook ne fournit aucun dataset
# (colonnes `text` / `sentiment` attendues mais jamais chargées).
# Section optionnelle laissée de côté faute de données sources.


### Créez une fonction pour nettoyer les mots d'arrêt et la ponctuation

Vous pouvez appeler votre fonction `func_clean`.
Votre fonction doit prendre un `str` en tant que paramètre unique et retourner un `str`.

Par exemple:

`func_clean("Hello, how are you? Fine, thank you.")`

`>>> 'hello fine thank'`

### Appliquer cette fonction

Apply cette fonction à `X` et stockez le résultat dans la variable ` X_clean`.

### Split de test de train

Divisez vos données `X_clean` et `y` avec un `train_test_split()`, et le même `random_state = 32`.

### tfidfvectorizer

- Initialiser votre vectorisateur et stockez le résultat dans `vectorizer`
- Train `vectorizer` sur ` x_train`.
- Train et Transform `x_train` avec votre vectorisateur et stockez le résultat dans `x_train_vecto`.
- Transformez `x_test` avec votre vectorisateur et stockez le résultat dans` x_test_vecto`.

### Régression logistique

Entrainer un modèle de régression logistique sur `x_train_vecto` et` y_train`.

Veuillez comparer les scores de précision des ensembles de formation et de test.Y a-t-il un sur-ajustement (`OverFitting)`?

Affichez également une matrice de confusion pour l'ensemble de tests.Combien de «mauvais» commentaires sont correctement prédits?

### Arbre de décision
Entrainer un modèle d'arbre de décision sur `x_train_vecto` et` y_train`.

Veuillez comparer les scores de précision des ensembles de formation et de test.Y a-t-il un sur-ajustement?Les scores sont-ils meilleurs qu'auparavant?

### Comparaison des algorithmes

Comparer les scores des 2 algorithmes, intéprétez et dites-nous lequel est meilleur selon vous ?

## Optionnel: algorithme `json` et manipulation.
Il s'agit d'un fichier JSON contenant plusieurs clés.
Chaque clé a une valeur, qui pourrait potentiellement être une autre clé, contenant une autre valeur, qui pourrait potentiellement être une autre clé, etc.

In [21]:
food = {
  "clé1": {
    "fruit1": "pomme",
    "légume4": "brocoli"
  },
  "clé2": {
    "légume1": "carotte",
    "fruit5": "banane",
    "légume3": "courgette"
  },
  "clé3": {
    "niveau1": {
      "niveau2": {
        "fruit3": "orange",
        "légume5": "aubergine",
        "fruit5": "mangue"
      }
    }
  },
  "clé4": {
    "niveau1": {
      "niveau2": {
        "niveau3": {
          "fruit6": "raisin",
          "fruit7": "fraise",
          "légume4": "poivron",
          "fruit2": "pastèque"
        }
      }
    }
  }
}


Problème:
Les fruits et légumes ont été égarés dans ce fichier JSON.L'objectif est de récupérer chacun des fruits et légumes et de les attribuer à deux listes correspondantes: `Fruits_list` &` LEGETALS_LIST`.

Solution attendue:

`fruits_list` = `['pomme', 'banane', 'orange', 'mangue', 'raisin', 'fraise', 'pastèque']`

`vegetables_list` = `['brocoli', 'carotte', 'courgette', 'aubergine', 'poivron']`


In [22]:
fruits_list = []
vegetables_list = []


def parcourir(noeud):
    """Parcourt récursivement le dictionnaire et trie fruits / légumes."""
    for cle, valeur in noeud.items():
        if isinstance(valeur, dict):
            parcourir(valeur)            # sous-dictionnaire -> on descend
        elif "fruit" in cle:
            fruits_list.append(valeur)
        elif "légume" in cle:
            vegetables_list.append(valeur)


parcourir(food)
print("fruits_list =", fruits_list)
print("vegetables_list =", vegetables_list)


fruits_list = ['pomme', 'banane', 'orange', 'mangue', 'raisin', 'fraise', 'pastèque']
vegetables_list = ['brocoli', 'carotte', 'courgette', 'aubergine', 'poivron']


Ensuite, vous créerai un nouveau dictionnaire, qui contiendra simplement deux clés: «fruits» et «légumes».Chaque clé aura la valeur de la liste des fruits et la liste des légumes.De cette façon, tout sera en ordre.

Solution attendue:

`food_dict` = `{'fruits': ['pomme','banane','orange','mangue','raisin','fraise','pastèque'],
 'legumes': ['brocoli', 'carotte', 'courgette', 'aubergine', 'poivron']}`

In [23]:
food_dict = {"fruits": fruits_list, "legumes": vegetables_list}
food_dict


{'fruits': ['pomme',
  'banane',
  'orange',
  'mangue',
  'raisin',
  'fraise',
  'pastèque'],
 'legumes': ['brocoli', 'carotte', 'courgette', 'aubergine', 'poivron']}